# 04_explainability — SHAP, Error Analysis & Threshold Tuning
**Week 3 | Telco Customer Churn**

Covers:
1. Load champion model
2. SHAP summary plot (global feature importance)
3. SHAP waterfall plot (single prediction explanation)
4. Segment error analysis — false negatives among high-CLV customers
5. Cost-sensitive threshold tuning
6. Model Card

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import get_splits, get_preprocessor_pipeline
from src.evaluate import evaluate, print_report, cost_matrix_threshold, best_threshold

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Champion Model & Data

In [ ]:
with open('../models/churn_model.pkl', 'rb') as f:
    bundle = pickle.load(f)

model        = bundle['model']
preprocessor = bundle['preprocessor']

print(f'Loaded model: {type(model).__name__}')

# Raw splits (needed for segment analysis)
X_train_raw, X_test_raw, y_train, y_test = get_splits(
    path='../data/processed/telco_churn_cleaned.csv'
)

X_train_t = preprocessor.transform(X_train_raw)
X_test_t  = preprocessor.transform(X_test_raw)

y_prob = model.predict_proba(X_test_t)[:, 1]
y_pred = model.predict(X_test_t)

metrics = evaluate(y_test, y_pred, y_prob, label=type(model).__name__)
print(metrics)

## 2. Get Feature Names After OneHot Encoding

In [ ]:
from src.features import CATEGORICAL_COLS, NUMERIC_COLS

ohe = preprocessor.named_transformers_['ohe']
ohe_feature_names = ohe.get_feature_names_out(CATEGORICAL_COLS).tolist()
feature_names = ohe_feature_names + NUMERIC_COLS

print(f'Total features after encoding: {len(feature_names)}')
print(feature_names[:10], '...')

## 3. SHAP — Global Feature Importance (Summary Plot)

In [ ]:
# Use TreeExplainer for tree-based models, KernelExplainer for LR
import sklearn

model_type = type(model).__name__

if model_type in ('RandomForestClassifier', 'XGBClassifier', 'LGBMClassifier'):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_t)
    # For binary classifiers, shap_values may be a list [class0, class1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]   # class 1 = Churn
    else:
        shap_vals = shap_values
else:
    # Logistic Regression fallback
    explainer = shap.LinearExplainer(model, X_train_t)
    shap_vals = explainer.shap_values(X_test_t)

print(f'SHAP values shape: {shap_vals.shape}')

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals,
    X_test_t,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Summary Plot — Top 20 Features')
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/shap_summary.png')

## 4. SHAP — Bar Plot (Mean |SHAP|)

In [ ]:
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_df = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_shap})
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=shap_df, x='mean_abs_shap', y='feature', palette='Blues_r')
plt.title('Top 15 Features by Mean |SHAP| Value')
plt.xlabel('Mean |SHAP| value')
plt.tight_layout()
plt.savefig('../reports/shap_bar.png', dpi=150)
plt.show()
print('Saved → reports/shap_bar.png')
print('\nTop 10 drivers:')
print(shap_df.head(10).to_string(index=False))

## 5. SHAP — Waterfall Plot (Single High-Risk Customer)

In [ ]:
# Pick the test customer with the highest predicted churn probability
high_risk_idx = np.argmax(y_prob)
print(f'High-risk customer index: {high_risk_idx}')
print(f'Predicted churn prob: {y_prob[high_risk_idx]:.3f}')
print(f'Actual label: {"Churn" if y_test.iloc[high_risk_idx] == 1 else "No Churn"}')

# Build SHAP Explanation object for waterfall
if model_type in ('RandomForestClassifier', 'XGBClassifier', 'LGBMClassifier'):
    exp = shap.Explanation(
        values=shap_vals[high_risk_idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
        data=X_test_t[high_risk_idx],
        feature_names=feature_names
    )
else:
    exp = shap.Explanation(
        values=shap_vals[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_test_t[high_risk_idx],
        feature_names=feature_names
    )

plt.figure()
shap.waterfall_plot(exp, max_display=15, show=False)
plt.title('SHAP Waterfall — Highest-Risk Customer')
plt.tight_layout()
plt.savefig('../reports/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/shap_waterfall.png')

## 6. Segment Error Analysis — False Negatives Among High-CLV Customers

**Business question:** Does the model miss long-tenure, high-value churners?  
These are the *costliest* false negatives — customers we lose without offering retention.

In [ ]:
# Rebuild test set with original feature values for segment analysis
test_df = X_test_raw.copy()
test_df['y_true']  = y_test.values
test_df['y_pred']  = y_pred
test_df['y_prob']  = y_prob

# CLV = MonthlyCharges × tenure  (already in cleaned data)
test_df['CLV'] = test_df['MonthlyCharges'] * test_df['tenure']

# Error type
def error_type(row):
    if row['y_true'] == 1 and row['y_pred'] == 0:
        return 'FN'   # missed churner — costliest
    elif row['y_true'] == 0 and row['y_pred'] == 1:
        return 'FP'   # false alarm
    elif row['y_true'] == 1 and row['y_pred'] == 1:
        return 'TP'
    else:
        return 'TN'

test_df['error'] = test_df.apply(error_type, axis=1)

print('Error distribution:')
print(test_df['error'].value_counts())

In [ ]:
# CLV distribution by error type
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=test_df[test_df['error'].isin(['FN', 'TP', 'FP', 'TN'])],
    x='error', y='CLV',
    order=['TP', 'FN', 'FP', 'TN'],
    palette={'TP': '#2ecc71', 'FN': '#e74c3c', 'FP': '#f39c12', 'TN': '#95a5a6'}
)
plt.title('CLV Distribution by Prediction Error Type')
plt.ylabel('Customer Lifetime Value (₹)')
plt.xlabel('Prediction Type')
plt.tight_layout()
plt.savefig('../reports/error_clv_boxplot.png', dpi=150)
plt.show()
print('Saved → reports/error_clv_boxplot.png')

In [ ]:
# High-CLV false negatives — the most dangerous misses
clv_75th = test_df['CLV'].quantile(0.75)
high_clv_fn = test_df[(test_df['error'] == 'FN') & (test_df['CLV'] >= clv_75th)]
all_fn      = test_df[test_df['error'] == 'FN']

print(f'CLV 75th percentile: ₹{clv_75th:.0f}')
print(f'Total False Negatives: {len(all_fn)}')
print(f'High-CLV False Negatives (CLV ≥ 75th pct): {len(high_clv_fn)}')
print(f'  → {len(high_clv_fn)/len(all_fn):.1%} of all FNs are high-CLV customers')
print(f'\nHigh-CLV FN profile:')
print(high_clv_fn[['tenure', 'MonthlyCharges', 'CLV', 'Contract', 'y_prob']].describe().round(2))

In [ ]:
# Tenure distribution: FN vs TP (among actual churners)
churners = test_df[test_df['y_true'] == 1].copy()

plt.figure(figsize=(10, 5))
sns.histplot(
    data=churners, x='tenure', hue='error',
    hue_order=['TP', 'FN'],
    palette={'TP': '#2ecc71', 'FN': '#e74c3c'},
    bins=20, kde=True
)
plt.title('Tenure Distribution: Correctly Caught (TP) vs Missed (FN) Churners')
plt.xlabel('Tenure (months)')
plt.tight_layout()
plt.savefig('../reports/error_tenure_dist.png', dpi=150)
plt.show()
print('Saved → reports/error_tenure_dist.png')

## 7. Cost-Sensitive Threshold Tuning

**Cost model:**
- Offering retention to a churner (TP): costs ₹100 (retention offer)
- Offering retention to a non-churner (FP): costs ₹100 (wasted offer)
- Missing a churner (FN): costs ₹500 (avg CLV lost)
- Correctly ignoring a non-churner (TN): ₹0

Default threshold = 0.5 is not optimal. We tune it to minimise total business cost.

In [ ]:
RETENTION_COST = 100   # ₹ per offer sent
AVG_CLV_LOST   = 500   # ₹ average CLV of a churner

cost_df = cost_matrix_threshold(
    y_test.values, y_prob,
    retention_cost=RETENTION_COST,
    avg_clv_lost=AVG_CLV_LOST
)

opt_threshold = best_threshold(cost_df)
print(f'Optimal threshold (min cost): {opt_threshold}')
print(f'Default threshold (0.5) cost: ₹{cost_df[cost_df.threshold==0.5]["total_cost"].values[0]:,.0f}')
print(f'Optimal threshold cost:       ₹{cost_df[cost_df.threshold==opt_threshold]["total_cost"].values[0]:,.0f}')

In [ ]:
# Plot cost vs threshold
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(cost_df['threshold'], cost_df['total_cost'], color='#e74c3c', lw=2, label='Total Cost (₹)')
ax1.axvline(opt_threshold, color='#e74c3c', linestyle='--', label=f'Optimal threshold = {opt_threshold}')
ax1.axvline(0.5, color='gray', linestyle=':', label='Default threshold = 0.5')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Total Business Cost (₹)', color='#e74c3c')

ax2 = ax1.twinx()
ax2.plot(cost_df['threshold'], cost_df['f1'],     color='#2ecc71', lw=2, label='F1')
ax2.plot(cost_df['threshold'], cost_df['recall'], color='#3498db', lw=2, linestyle='--', label='Recall')
ax2.set_ylabel('Score', color='#2ecc71')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.title('Cost-Sensitive Threshold Tuning')
plt.tight_layout()
plt.savefig('../reports/threshold_tuning.png', dpi=150)
plt.show()
print('Saved → reports/threshold_tuning.png')

In [ ]:
# Compare default vs optimal threshold
y_pred_opt = (y_prob >= opt_threshold).astype(int)
metrics_opt = evaluate(y_test, y_pred_opt, y_prob, label=f'threshold={opt_threshold}')

print('\n--- Default threshold (0.5) ---')
print_report(y_test, y_pred)

print(f'\n--- Optimal threshold ({opt_threshold}) ---')
print_report(y_test, y_pred_opt)

print('\nMetrics comparison:')
comparison = pd.DataFrame([metrics, metrics_opt])
print(comparison.to_string(index=False))

## 8. Model Card

In [ ]:
model_card = f"""
# Model Card — Telco Customer Churn Prediction

## Model Details
- **Model type:** {type(model).__name__}
- **Task:** Binary classification (Churn = 1, No Churn = 0)
- **Training data:** Telco Customer Churn dataset (Kaggle), 7,043 customers
- **Features:** 22 input features → {X_test_t.shape[1]} after OneHot encoding

## Performance (Test Set, default threshold 0.5)
- ROC-AUC:  {metrics['roc_auc']}
- F1:       {metrics['f1']}
- Precision:{metrics['precision']}
- Recall:   {metrics['recall']}

## Performance (Optimal threshold {opt_threshold})
- ROC-AUC:  {metrics_opt['roc_auc']}
- F1:       {metrics_opt['f1']}
- Precision:{metrics_opt['precision']}
- Recall:   {metrics_opt['recall']}

## Top Churn Drivers (SHAP)
{shap_df.head(5)[['feature','mean_abs_shap']].to_string(index=False)}

## Cost Model
- Retention offer cost: ₹{RETENTION_COST} per customer
- Average CLV lost per missed churner: ₹{AVG_CLV_LOST}
- Optimal threshold minimises total business cost

## Limitations
- Model trained on historical data; performance may degrade if customer behaviour shifts
- High-CLV long-tenure churners are harder to detect (see error analysis)
- CLV formula is simplified (MonthlyCharges × tenure); does not account for future revenue

## Intended Use
- Identify high-risk customers for targeted retention campaigns
- Prioritise outreach by predicted churn probability × CLV
- NOT intended for automated contract termination decisions
"""

with open('../reports/model_card.md', 'w') as f:
    f.write(model_card)

print(model_card)
print('Saved → reports/model_card.md')